In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import random
import types

from time import time

from scipy.special import comb, loggamma, lambertw
from scipy.stats import multinomial, expon

from silence_tensorflow import silence_tensorflow
silence_tensorflow()
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow import keras
from tensorflow.keras import optimizers, initializers, regularizers, layers

config = tf.compat.v1.ConfigProto()
config.gpu_options.allow_growth = True
sess = tf.compat.v1.Session(config = config)

import os, shutil
from pathlib import Path
import json
import subprocess

import lifelines
from lifelines.utils import concordance_index
from lifelines.statistics import logrank_test

import sys
sys.path.insert(0, "../src")
from package import cure_aft
from package import gndr_utils as utils
from package import evaluation_metrics as evaluations

import thetaflow as thf

E0000 00:00:1793376916.205160   11603 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793376916.210335   11603 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1793376916.224047   11603 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793376916.224069   11603 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793376916.224072   11603 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1793376916.224075   11603 computation_placer.cc:177] computation placer already registered. Please check linka

In [2]:
def load_cv_train_test(fold):
    df_train = pd.read_csv("../data/01_mlready/Cross Validation/{}_fold_train.csv".format(fold))
    df_test = pd.read_csv("../data/01_mlready/Cross Validation/{}_fold_test.csv".format(fold))

    # Lifetimes and censoring indicators
    y_train = df_train["time"] / 12
    delta_train = df_train["delta"]
    # Data for the neural network interpreter
    X_train_nn = df_train.iloc[:, 10:]
    # For all parameters from the AFT model, consider the same set of features as predictors
    z_train_mu = df_train.iloc[:, 3:10]
    z_train_mu.insert(0, "const", 1)
    z_train_sigma = df_train.iloc[:, 5:10]
    z_train_sigma.insert(0, "const", 1)
    z_train_p = df_train.iloc[:, 3:10]
    z_train_p.insert(0, "const", 1)
    
    # Lifetimes and censoring indicators
    y_test = df_test["time"] / 12
    delta_test = df_test["delta"]
    # Data for the neural network interpreter
    X_test_nn = df_test.iloc[:, 10:]
    # For all parameters from the AFT model, consider the same set of features as predictors
    z_test_mu = df_test.iloc[:, 3:10]
    z_test_mu.insert(0, "const", 1)
    z_test_sigma = df_test.iloc[:, 5:10]
    z_test_sigma.insert(0, "const", 1)
    z_test_p = df_test.iloc[:, 3:10]
    z_test_p.insert(0, "const", 1)

    return y_train, delta_train, X_train_nn, z_train_mu, z_train_sigma, z_train_p, \
           y_test, delta_test, X_test_nn, z_test_mu, z_test_sigma, z_test_p

In [3]:
def load_fold_model(structure, dist, fold):
    # Load the data corresponding to the desired fold
    y_train, delta_train, X_train_nn, z_train_mu, z_train_sigma, z_train_p, \
    y_test, delta_test, X_test_nn, z_test_mu, z_test_sigma, z_test_p = load_cv_train_test(fold)
    print("Struct: {} - Dist: {} - Fold: {}".format(structure, dist, fold))
    
    if(structure == 1):
        model_parameters, model_loss, model_survival_cure = cure_aft.build_cure_aft_structure1(dist, z_train_mu, z_train_sigma, z_train_p)
    elif(structure == 2):
        # Remove the first column (intercept) from z_mu_train and z_mu_test
        z_train_mu = z_train_mu.iloc[:,1:]
        z_test_mu = z_test_mu.iloc[:,1:]
        
        model_parameters, model_loss, model_survival_cure = cure_aft.build_cure_aft_structure2(dist, z_train_mu, z_train_sigma, z_train_p)
        # The neural network architecture here considers one output
        nn_arch = cure_aft.neural_network1
    elif(structure == 3):
        # Remove the first column (intercept) from z_p_train and z_p_test
        z_train_p = z_train_p.iloc[:,1:]
        z_test_p = z_test_p.iloc[:,1:]
        
        model_parameters, model_loss, model_survival_cure = cure_aft.build_cure_aft_structure3(dist, z_train_mu, z_train_sigma, z_train_p)
        # The neural network architecture here considers one output
        nn_arch = cure_aft.neural_network1
    elif(structure == 4):
        # Remove the first column (intercept) from z_mu_train, z_mu_test, z_p_train and z_p_test
        z_train_mu = z_train_mu.iloc[:,1:]
        z_test_mu = z_test_mu.iloc[:,1:]
        z_train_p = z_train_p.iloc[:,1:]
        z_test_p = z_test_p.iloc[:,1:]
        
        model_parameters, model_loss, model_survival_cure = cure_aft.build_cure_aft_structure4(dist, z_train_mu, z_train_sigma, z_train_p)
        # The neural network architecture here considers two outputs
        nn_arch = cure_aft.neural_network2
    
    if(structure == 1):
        model = thf.ModelNN(model_parameters, model_loss, seed = 10)
    else:
        model = thf.ModelNN(model_parameters, model_loss,
                            nn_arch, cure_aft.network_call, cure_aft.network_call_nolast,
                            input_dim = (X_train_nn.shape[1], ), seed = 10)
    
    # Load the model weights and attributes
    model_filename = "{}_model{}_fold_{}".format(dist, structure, fold)
    if(structure == 1):
        model(None)
        
    model.load_model( "../model_weights/Cross Validation/{}".format(model_filename) )
    
    if(structure == 1):
        model_summary = model_survival_cure(model,
                                            y_train, delta_train, z_train_mu, z_train_sigma, z_train_p,
                                            y_test, delta_test, z_test_mu, z_test_sigma, z_test_p, ngrid = 100)
    else:
        model_summary = model_survival_cure(model,
                                            y_train, delta_train, X_train_nn, z_train_mu, z_train_sigma, z_train_p,
                                            y_test, delta_test, X_test_nn, z_test_mu, z_test_sigma, z_test_p, ngrid = 100)
    return model, model_summary

In [4]:
distributions = ["wb", "ln", "ll"]
folds = [1,2,3,4,5]
structures = [1,2,3,4]

model_list = {}
summary_list = {}

for structure in structures:
    summary_list[structure] = {}
    model_list[structure] = {}
    for dist in distributions:
        summary_list[structure][dist] = []
        model_list[structure][dist] = []
        for fold in folds:
            wb_model, wb_model_summary = load_fold_model(structure, dist, fold)
            model_list[structure][dist].append( wb_model )
            summary_list[structure][dist].append( wb_model_summary )

Struct: 1 - Dist: wb - Fold: 1
Model successfully loaded from ../model_weights/Cross Validation/wb_model1_fold_1.
Struct: 1 - Dist: wb - Fold: 2
Model successfully loaded from ../model_weights/Cross Validation/wb_model1_fold_2.
Struct: 1 - Dist: wb - Fold: 3
Model successfully loaded from ../model_weights/Cross Validation/wb_model1_fold_3.
Struct: 1 - Dist: wb - Fold: 4
Model successfully loaded from ../model_weights/Cross Validation/wb_model1_fold_4.
Struct: 1 - Dist: wb - Fold: 5
Model successfully loaded from ../model_weights/Cross Validation/wb_model1_fold_5.
Struct: 1 - Dist: ln - Fold: 1
Model successfully loaded from ../model_weights/Cross Validation/ln_model1_fold_1.
Struct: 1 - Dist: ln - Fold: 2
Model successfully loaded from ../model_weights/Cross Validation/ln_model1_fold_2.
Struct: 1 - Dist: ln - Fold: 3
Model successfully loaded from ../model_weights/Cross Validation/ln_model1_fold_3.
Struct: 1 - Dist: ln - Fold: 4
Model successfully loaded from ../model_weights/Cross Val

### Evaluation Metrics

In [16]:
distributions = ["wb", "ln", "ll"]
folds = [1,2,3,4,5]
structures = [1,2,3,4]

# List of Anderson-Darling statistics
ad_list, ad_list_folds = {}, {}
# List of Integrated Brier Score metrics
ibs_list, ibs_list_folds = {}, {}

for structure in structures:
    ad_list_folds[structure] = {}
    ibs_list_folds[structure] = {}
    ad_list[structure] = {}
    ibs_list[structure] = {}
    for dist in distributions:
        ad_list_folds[structure][dist] = []
        ibs_list_folds[structure][dist] = []
        for i in range(len(folds)):
            ad_list_folds[structure][dist].append( float(evaluations.calculate_ad_statistic( summary_list[structure][dist][i] )) )
            ibs_list_folds[structure][dist].append( float(evaluations.calculate_integrated_ipcw_brier_score( summary_list[structure][dist][i] )) )
        ad_list[structure][dist] = np.mean( ad_list_folds[structure][dist] )
        ibs_list[structure][dist] = np.mean( ibs_list_folds[structure][dist] )

In [17]:
metrics_ibs = pd.DataFrame(ibs_list).T.stack().reset_index()

metrics = pd.DataFrame(ad_list).T.stack().reset_index()
metrics = pd.concat([metrics, metrics_ibs.iloc[:,-1]], axis = 1)

metrics.columns = ["scenario", "model", "AD", "IBS"]
metrics.sort_values(by = "AD")

,scenario,model,AD,IBS
2,1,ll,0.000933,0.211131
1,1,ln,0.001075,0.210571
7,3,ln,0.001140,0.214871
8,3,ll,0.001173,0.221315
0,1,wb,0.001433,0.211434
11,4,ll,0.001450,0.229731
6,3,wb,0.001613,0.213487
9,4,wb,0.002197,0.238018
10,4,ln,0.002220,0.218104
3,2,wb,0.003838,0.222179


In [18]:
metrics.sort_values(by = "IBS")

,scenario,model,AD,IBS
1,1,ln,0.001075,0.210571
2,1,ll,0.000933,0.211131
0,1,wb,0.001433,0.211434
6,3,wb,0.001613,0.213487
7,3,ln,0.001140,0.214871
10,4,ln,0.002220,0.218104
4,2,ln,0.005043,0.219328
5,2,ll,0.005676,0.220613
8,3,ll,0.001173,0.221315
3,2,wb,0.003838,0.222179
